Para simular um contexto real de engenharia de dados aplicado ao futebol, foi desenvolvido um processo ETL (Extract, Transform, Load), responsável pela extração, transformação e carregamento dos dados para uma base de dados PostgreSQL. Esta abordagem permite estruturar os dados de forma consistente e persistente, facilitando posteriormente a sua exploração analítica e integração em aplicações e dashboards.

O processo ETL é iniciado na etapa de extração dos dados da StatsBomb através da respetiva API. Todo o código desta fase foi desenvolvido de forma modular e parametrizável, permitindo que a integração de novas competições possa ser realizada de forma simples, escalável e com alterações mínimas ao pipeline existente.

# Extract

In [9]:
# import das bibliotecas necessárias
import pandas as pd
import numpy as np
from statsbombpy import sb
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

In [10]:
# função para obter competições
def get_competitions(competition_season_pairs):
    comps_df = sb.competitions()

    mask = False

    for competition_name, season_name in competition_season_pairs:
        mask |= (
            (comps_df["competition_name"] == competition_name) &
            (comps_df["season_name"] == season_name)
        )

    return comps_df[mask]

Devido às limitações de disponibilidade dos dados open data da StatsBomb, que incluem poucas competições completas com dados recentes, foi selecionado o Euro 2024. Esta competição apresenta uma cobertura completa e atual, tornando-se adequada para a aplicação e avaliação da métrica desenvolvida.

In [11]:
competitions = [
    #("1. Bundesliga", "2023/2024"),
    ("UEFA Euro", "2024")
]

competitions_df = get_competitions(competitions)

# guardar os dados raw das competições
competitions_df.to_parquet("../data/raw/competitions.parquet", index=False)
competitions_df

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
73,55,282,Europe,UEFA Euro,male,False,True,2024,2026-05-01T19:54:25.846072,2026-05-01T19:58:06.077979,2026-05-01T19:58:06.077979,2026-05-01T19:54:25.846072


In [12]:
matches_dfs = []

# iterar sobre as competições e obter os jogos
for _, row in competitions_df.iterrows():
    matches = sb.matches(

        # adicionar colunas de competition_id e season_id para ligação entre tabelas
        competition_id=row["competition_id"],
        season_id=row["season_id"]
    )

    matches_dfs.append(matches)

matches_df = pd.concat(matches_dfs, ignore_index=True)

#guardar os dados raw dos jogos
matches_df.to_parquet("../data/raw/matches.parquet", index=False)
matches_df

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,home_manager_country_name,away_manager_id,away_manager_name,away_manager_nickname,away_manager_dob,away_manager_country_id,away_manager_country_name,data_version,shot_fidelity_version,xy_fidelity_version
0,3930166,2024-06-18,19:00:00.000,2,1,available,available,2026-05-01T19:52:36.631184,2026-05-01T19:56:35.877702,1,...,Spain,1000468,Ivan Hašek,None,1963-09-06,60,Czech Republic,1.1.0,2,2
1,3930159,2024-06-15,13:00:00.000,1,3,available,available,2026-05-01T19:45:59.392947,2026-05-01T19:50:52.970161,1,...,Italy,2832,Murat Yakin,None,1974-09-15,221,Switzerland,1.1.0,2,2
2,3930163,2024-06-16,19:00:00.000,0,1,available,available,2026-05-01T19:48:19.783129,2026-05-01T19:49:47.226912,1,...,Serbia,277,Gareth Southgate,None,1970-09-03,68,England,1.1.0,2,2
3,3930181,2024-06-25,19:00:00.000,0,0,available,available,2026-05-01T19:48:27.887397,2026-05-01T19:50:03.279607,3,...,England,1733,Matjaž Kek,None,1961-09-09,208,Slovenia,1.1.0,2,2
4,3930178,2024-06-24,19:00:00.000,1,1,available,available,2026-05-01T19:45:23.956799,2026-05-01T19:46:35.413328,3,...,Croatia,399,Luciano Spalletti,None,1959-03-07,112,Italy,1.1.0,2,2
5,3930160,2024-06-15,16:00:00.000,3,0,available,available,2026-05-01T19:45:01.052020,2026-05-01T19:46:14.941863,1,...,Spain,307,Zlatko Dalić,None,1966-10-26,56,Croatia,1.1.0,2,2
6,3930170,2024-06-20,13:00:00.000,1,1,available,available,2026-05-01T19:50:11.398180,2026-05-01T19:56:14.983173,2,...,Slovenia,5908,Dragan Stojković,None,1965-03-03,203,Serbia,1.1.0,2,2
7,3938643,2024-06-25,16:00:00.000,1,1,available,available,2026-05-01T09:32:36.453598,2026-05-01T09:37:02.207863,3,...,France,363,Michał Probierz,None,1972-09-24,182,Poland,1.1.0,2,2
8,3938641,2024-06-21,16:00:00.000,1,3,available,available,2026-05-01T09:29:24.560133,2026-05-01T09:30:43.170204,2,...,Poland,380,Ralf Rangnick,None,1958-06-29,85,Germany,1.1.0,2,2
9,3938642,2024-06-22,13:00:00.000,1,1,available,available,2026-05-01T09:31:28.135473,2026-05-01T09:32:48.737879,2,...,France,1000468,Ivan Hašek,None,1963-09-06,60,Czech Republic,1.1.0,2,2


In [13]:
all_lineups = []

# iterar sobre os jogos para obter os lineups
for _, match in matches_df.iterrows():

    match_id = match["match_id"]

    lineup_dict = sb.lineups(match_id=match_id)

    team_mapping = {
        match["home_team"]: match["home_team_id"],
        match["away_team"]: match["away_team_id"]
    }

    for team_name, lineup_df in lineup_dict.items():

        lineup_df = lineup_df.copy()

        lineup_df["team_name"] = team_name
        lineup_df["team_id"] = team_mapping.get(team_name)
        lineup_df["match_id"] = match_id

        all_lineups.append(lineup_df)

lineups_df = pd.concat(all_lineups, ignore_index=True)

# guardar os dados raw dos lineups
lineups_df.to_parquet("../data/raw/lineups.parquet", index=False)
lineups_df

,player_id,player_name,player_nickname,jersey_number,country,cards,positions,team_name,team_id,match_id
0,3193,Bernardo Mota Veiga de Carvalho e Silva,Bernardo Silva,10,Portugal,[],"[{'position_id': 19, 'position': 'Center Attac...",Portugal,780,3930166
1,5204,Bruno Miguel Borges Fernandes,Bruno Fernandes,8,Portugal,[],"[{'position_id': 11, 'position': 'Left Defensi...",Portugal,780,3930166
2,5205,Rui Pedro dos Santos Patrício,Rui Patrício,1,Portugal,[],[],Portugal,780,3930166
3,5206,Rúben Santos Gato Alves Dias,Rúben Dias,4,Portugal,[],"[{'position_id': 3, 'position': 'Right Center ...",Portugal,780,3930166
4,5207,Cristiano Ronaldo dos Santos Aveiro,Cristiano Ronaldo,7,Portugal,[],"[{'position_id': 22, 'position': 'Right Center...",Portugal,780,3930166
...,...,...,...,...,...,...,...,...,...,...
2582,22308,Anthony Ralston,None,2,Scotland,"[{'time': '47:19', 'card_type': 'Yellow Card',...","[{'position_id': 7, 'position': 'Right Wing Ba...",Scotland,942,3930158
2583,22379,Angus Gunn,None,1,Scotland,[],"[{'position_id': 1, 'position': 'Goalkeeper', ...",Scotland,942,3930158
2584,31067,Billy Gilmour,None,14,Scotland,[],"[{'position_id': 10, 'position': 'Center Defen...",Scotland,942,3930158
2585,38169,Lawrence Shankland,None,9,Scotland,[],"[{'position_id': 23, 'position': 'Center Forwa...",Scotland,942,3930158


In [14]:
# função para obter eventos com base em lista de match_ids
def get_all_events(match_ids):

    all_events = []

    for match_id in tqdm(match_ids):

        try:
            events_df = sb.events(match_id=match_id)

            # events_df["home_team"] = match["home_team"]
            # events_df["away_team"] = match["away_team"]
            # events_df["competition"] = match.get("competition_name", None)
            # events_df["season"] = match.get("season_name", None)

            all_events.append(events_df)

        except Exception as e:
            print(f"Error loading {match_id}: {e}")

    return pd.concat(all_events, ignore_index=True)


events_df = get_all_events(matches_df["match_id"])
events_df

100%|██████████| 51/51 [00:33<00:00,  1.51it/s]


,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,foul_committed_penalty,foul_won_penalty,pass_deflected,shot_redirect,goalkeeper_punched_out,shot_open_goal,block_save_block,goalkeeper_success_in_play,goalkeeper_penalty_saved_to_post,shot_follows_dribble
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187919,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187920,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187922,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# função para obter dados 360 com base em lista de match_ids
def get_all_frames(match_ids):

    all_frames = []

    for match_id in tqdm(match_ids):

        try:
            frames_df = sb.frames(match_id=match_id)

            all_frames.append(frames_df)

        except Exception as e:
            print(f"Error loading {match_id}: {e}")

    return pd.concat(all_frames, ignore_index=True)

events_360_df = get_all_frames(matches_df["match_id"])
events_360_df

100%|██████████| 51/51 [01:11<00:00,  1.41s/it]


,id,visible_area,match_id,teammate,actor,keeper,location
0,87a790b7-17a6-4bbd-953f-a06c07edbad9,"[18.400697860849267, 80.0, 46.48865423106284, ...",3930166,True,False,False,"[38.20338261131079, 29.491675823953365]"
1,87a790b7-17a6-4bbd-953f-a06c07edbad9,"[18.400697860849267, 80.0, 46.48865423106284, ...",3930166,True,False,False,"[40.435487610014256, 49.496164383322295]"
2,87a790b7-17a6-4bbd-953f-a06c07edbad9,"[18.400697860849267, 80.0, 46.48865423106284, ...",3930166,True,False,False,"[51.887832712562904, 56.43639891583137]"
3,87a790b7-17a6-4bbd-953f-a06c07edbad9,"[18.400697860849267, 80.0, 46.48865423106284, ...",3930166,True,False,False,"[60.036085232306185, 31.006609171941484]"
4,87a790b7-17a6-4bbd-953f-a06c07edbad9,"[18.400697860849267, 80.0, 46.48865423106284, ...",3930166,True,False,False,"[60.838859912993875, 4.272189145609488]"
...,...,...,...,...,...,...,...
2698994,5ce3666b-f6c5-4426-a16c-7928236bbdac,"[0.0, 80.0, 0.0, 28.23202227625909, 22.9743547...",3930158,True,False,False,"[15.948856551775805, 55.95216286228902]"
2698995,5ce3666b-f6c5-4426-a16c-7928236bbdac,"[0.0, 80.0, 0.0, 28.23202227625909, 22.9743547...",3930158,False,False,False,"[17.452580090224245, 33.005712133560706]"
2698996,5ce3666b-f6c5-4426-a16c-7928236bbdac,"[0.0, 80.0, 0.0, 28.23202227625909, 22.9743547...",3930158,False,False,False,"[19.947396309933282, 45.41233704070139]"
2698997,5ce3666b-f6c5-4426-a16c-7928236bbdac,"[0.0, 80.0, 0.0, 28.23202227625909, 22.9743547...",3930158,True,False,False,"[20.328016357383106, 42.49520526434236]"


In [16]:
# eliminar coluna para evitar conflitos de duplicação de colunas após o merge
events_360_df = events_360_df.drop(columns=["match_id"], errors="ignore")

# realizamos o merge dos eventos com os dados 360º através do id do evento
events_df = events_df.merge(events_360_df, on="id", how="left")

# guardar os dados raw dos eventos (eventos + 360)
events_df.to_parquet("../data/raw/events.parquet", index=False)
events_df

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,shot_open_goal,block_save_block,goalkeeper_success_in_play,goalkeeper_penalty_saved_to_post,shot_follows_dribble,visible_area,teammate,actor,keeper,location_y
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2722388,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"[29.752411154964918, 79.22086739213373, 0.0, 4...",False,False,False,"[17.191577570485677, 42.22972347817744]"
2722389,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"[29.752411154964918, 79.22086739213373, 0.0, 4...",True,False,False,"[17.965696465999017, 29.694118095738418]"
2722390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"[29.752411154964918, 79.22086739213373, 0.0, 4...",True,False,False,"[25.70225726023301, 36.307377809078865]"
2722391,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,"[29.752411154964918, 79.22086739213373, 0.0, 4...",False,False,False,"[30.580730722937282, 47.8590938785301]"
